# Governance Tracker Update

Compares the latest inventory snapshot against `cleanup_tracker` to determine:
1. **New candidates** — items scoring above threshold, not yet tracked
2. **Resolved items** — previously flagged items that owners have actioned
3. **Escalations** — items due for the next warning level
4. **Deletion candidates** — items past 3 warnings with no action

Requires: access to the **DW_Fabric** Warehouse (Governance schema).

**No API calls. No admin permission. Reads and writes Warehouse tables only.**


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import uuid
import logging
from datetime import datetime, timezone, timedelta

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("tracker_update")

pipeline_run_id = str(uuid.uuid4())
run_timestamp   = datetime.now(timezone.utc).isoformat()

log.info(f"Pipeline run ID: {pipeline_run_id}")
log.info(f"Run timestamp:   {run_timestamp}")

# Warehouse connection (Governance schema in DW_Fabric)
import pyodbc, struct, notebookutils

WAREHOUSE_SQL_ENDPOINT = "<WAREHOUSE_SQL_ENDPOINT>"
WAREHOUSE_DATABASE     = "DW_Fabric"
GOVERNANCE_SCHEMA      = "Governance"

def get_warehouse_connection():
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};"
        f"Server={WAREHOUSE_SQL_ENDPOINT};"
        f"Database={WAREHOUSE_DATABASE};"
        f"Encrypt=Yes;TrustServerCertificate=No"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

conn = get_warehouse_connection()


## 2. Load configuration

In [ ]:
# Read governance config into a dictionary
df_config = pd.read_sql(f"SELECT config_key, config_value FROM {GOVERNANCE_SCHEMA}.governance_config", conn)
config = dict(zip(df_config["config_key"], df_config["config_value"]))

score_threshold      = int(config.get("cleanup_score_threshold", "30"))
warnings_before_del  = int(config.get("warnings_before_delete", "3"))
days_between_warn    = int(config.get("days_between_warnings", "1"))
protected_types      = [t.strip() for t in config.get("protected_types", "").split(",") if t.strip()]
protected_items      = [i.strip() for i in config.get("protected_items", "").split(",") if i.strip()]
enable_auto_delete   = config.get("enable_auto_delete", "false").lower() == "true"
dry_run              = config.get("dry_run", "true").lower() == "true"

log.info(f"Config loaded:")
log.info(f"  Score threshold:      {score_threshold}")
log.info(f"  Warnings before del:  {warnings_before_del}")
log.info(f"  Days between warns:   {days_between_warn}")
log.info(f"  Protected types:      {protected_types}")
log.info(f"  Protected items:      {len(protected_items)} items")
log.info(f"  Auto-delete enabled:  {enable_auto_delete}")
log.info(f"  Dry run:              {dry_run}")


## 3. Load latest inventory snapshot

In [ ]:
# Get the most recent snapshot
df_snapshot = pd.read_sql(f"""
    SELECT * FROM {GOVERNANCE_SCHEMA}.workspace_inventory_snapshot
    WHERE snapshot_id = (
        SELECT TOP 1 snapshot_id FROM {GOVERNANCE_SCHEMA}.workspace_inventory_snapshot
        ORDER BY snapshot_time_utc DESC
    )
""", conn)

log.info(f"Latest snapshot: {df_snapshot['snapshot_id'].iloc[0] if not df_snapshot.empty else 'NONE'}")
log.info(f"  Items in snapshot: {len(df_snapshot)}")

# Convert numeric columns
for col in ["cleanup_candidate_score", "is_stale", "is_unused_artifact",
            "has_missing_owner", "is_orphaned_model", "is_orphaned_endpoint",
            "is_duplicate_name", "days_since_modified", "days_since_last_used"]:
    if col in df_snapshot.columns:
        df_snapshot[col] = pd.to_numeric(df_snapshot[col], errors="coerce")


## 4. Load current tracker

In [ ]:
df_tracker = pd.read_sql(f"SELECT * FROM {GOVERNANCE_SCHEMA}.cleanup_tracker", conn)
log.info(f"Current tracker: {len(df_tracker)} items being tracked")

if not df_tracker.empty:
    status_counts = df_tracker["status"].value_counts()
    for status, count in status_counts.items():
        log.info(f"  {status}: {count}")


## 5. Determine actions

### Logic:
1. **New candidates** — in snapshot with score >= threshold, NOT in tracker (or previously resolved)
2. **Resolved** — in tracker but item was modified/used/deleted/score dropped
3. **Escalate** — in tracker, not resolved, enough days elapsed since last warning
4. **Protected** — skip items in protected types or protected items list
5. **Active-item guard** — auto-exempt if recently used or modified


In [ ]:
# ── Build sets for quick lookup ──────────────────────────
snapshot_ids = set(df_snapshot["id"].dropna().astype(str))

# Active tracker items (not resolved, not deleted, not exempted)
# deletion_ready and pending_deletion MUST be included here — otherwise an item that
# reaches deletion_ready (e.g. whenever enable_auto_delete=false or dry_run=true, which is
# the default safe-testing state) stops being recognized as 'already tracked' on the next
# run, and gets re-inserted as a brand-new 'new' row every single run without ever touching
# or removing the existing deletion_ready row — producing duplicate rows per item_id that
# accumulate indefinitely.
active_statuses = {"new", "warning_1", "warning_2", "warning_3", "deletion_ready", "pending_deletion"}
if not df_tracker.empty:
    active_tracker = df_tracker[df_tracker["status"].isin(active_statuses)]
    tracked_ids = set(active_tracker["item_id"].astype(str))
else:
    active_tracker = pd.DataFrame()
    tracked_ids = set()

# ── Action lists ───────────────────────────────────────
new_candidates = []       # Items to add to tracker
resolved_items = []       # Items to mark as resolved
escalations = []          # Items to escalate (next warning level)
deletion_candidates = []  # Items past all warnings
skipped_protected = 0
skipped_active_guard = 0

now = datetime.now(timezone.utc)

log.info(f"Processing...")
log.info(f"  Snapshot items: {len(snapshot_ids)}")
log.info(f"  Active tracked: {len(tracked_ids)}")


### 5a. Detect resolved items

In [ ]:
# Check each actively tracked item for resolution signals
for _, row in active_tracker.iterrows():
    item_id = str(row["item_id"])
    resolved = False
    reason = ""

    # Signal 1: Item no longer in snapshot (owner deleted it)
    if item_id not in snapshot_ids:
        resolved = True
        reason = "Item no longer in workspace (deleted by owner)"

    else:
        # Find the item in the current snapshot
        snap_row = df_snapshot[df_snapshot["id"].astype(str) == item_id]
        if snap_row.empty:
            resolved = True
            reason = "Item not found in snapshot"
        else:
            snap = snap_row.iloc[0]

            # Signal 2: Score dropped below threshold — the general-purpose check that
            # correctly covers every flag combination (stale, unused, orphaned, etc.), so an
            # item stays tracked until it's genuinely no longer a governance concern.
            current_score = snap.get("cleanup_candidate_score", 0)
            if pd.notna(current_score) and int(current_score) < score_threshold:
                resolved = True
                reason = f"Score dropped to {int(current_score)} (below threshold {score_threshold})"

            # Signal 3: Recently modified (within 14 days) — real owner activity
            elif pd.notna(snap.get("days_since_modified")) and float(snap["days_since_modified"]) < 14:
                resolved = True
                reason = f"Recently modified ({int(snap['days_since_modified'])} days ago)"

    if resolved:
        resolved_items.append({
            "item_id": item_id,
            "item_name": row.get("item_name", ""),
            "item_type": row.get("item_type", ""),
            "owner_email": row.get("owner_email", ""),
            "reason": reason,
        })

log.info(f"  Resolved items: {len(resolved_items)}")
for r in resolved_items[:5]:
    log.info(f"    {r['item_name']}: {r['reason']}")
if len(resolved_items) > 5:
    log.info(f"    ... and {len(resolved_items) - 5} more")


### 5b. Detect new candidates

In [ ]:
# Items in snapshot with score >= threshold that aren't actively tracked
resolved_ids = {r["item_id"] for r in resolved_items}

for _, snap in df_snapshot.iterrows():
    item_id = str(snap["id"])
    score = snap.get("cleanup_candidate_score", 0)
    if pd.isna(score):
        continue
    score = int(score)

    # Must meet score threshold
    if score < score_threshold:
        continue

    # Skip if already actively tracked (and not just resolved)
    if item_id in tracked_ids and item_id not in resolved_ids:
        continue

    # Skip protected types
    item_type = str(snap.get("type", ""))
    if item_type in protected_types:
        skipped_protected += 1
        continue

    # Skip protected items
    if item_id in protected_items:
        skipped_protected += 1
        continue

    # Active-item guard: skip if recently used or modified
    dsm = snap.get("days_since_modified")
    if pd.notna(dsm) and float(dsm) < 14:
        skipped_active_guard += 1
        continue

    dsu = snap.get("days_since_last_used")
    if pd.notna(dsu) and float(dsu) < 30:
        skipped_active_guard += 1
        continue

    new_candidates.append({
        "item_id": item_id,
        "item_name": str(snap.get("name", "")),
        "item_type": item_type,
        "owner_email": str(snap.get("created_by", "")),
        "cleanup_score": score,
        "workspace_id": str(snap.get("workspace_id", "")),
    })

log.info(f"  New candidates: {len(new_candidates)}")
log.info(f"  Skipped (protected): {skipped_protected}")
log.info(f"  Skipped (active guard): {skipped_active_guard}")


### 5c. Determine escalations

In [ ]:
# For items already in tracker (not resolved), check if they need escalation
resolved_ids_set = {r["item_id"] for r in resolved_items}

for _, row in active_tracker.iterrows():
    item_id = str(row["item_id"])

    # Skip if just resolved
    if item_id in resolved_ids_set:
        continue

    # Skip protected types
    if str(row.get("item_type", "")) in protected_types:
        continue

    # Determine current warning level and when last warning was sent
    current_status = row.get("status", "new")
    warning_count = int(row.get("warning_count", 0)) if pd.notna(row.get("warning_count")) else 0

    # Find the date of the last warning
    last_warning_date = None
    for wc in [warning_count, 3, 2, 1]:
        col = f"warning_{wc}_date"
        if col in row and pd.notna(row.get(col)) and str(row.get(col)).strip():
            try:
                last_warning_date = datetime.fromisoformat(str(row[col]).replace("Z", "+00:00"))
                break
            except (ValueError, TypeError):
                continue

    # Check if enough days have passed since last warning
    if last_warning_date:
        days_since_warning = (now - last_warning_date).days
        if days_since_warning < days_between_warn:
            continue  # Too soon for next warning

    # Determine next action
    if warning_count < warnings_before_del:
        # Escalate to next warning level
        next_warning = warning_count + 1
        escalations.append({
            "item_id": item_id,
            "item_name": row.get("item_name", ""),
            "item_type": row.get("item_type", ""),
            "owner_email": row.get("owner_email", ""),
            "cleanup_score": row.get("cleanup_score", 0),
            "current_warning": warning_count,
            "next_warning": next_warning,
            "next_status": f"warning_{next_warning}",
            "workspace_id": row.get("workspace_id", ""),
        })
    else:
        # Past all warnings — candidate for deletion
        deletion_candidates.append({
            "item_id": item_id,
            "item_name": row.get("item_name", ""),
            "item_type": row.get("item_type", ""),
            "owner_email": row.get("owner_email", ""),
            "cleanup_score": row.get("cleanup_score", 0),
            "workspace_id": row.get("workspace_id", ""),
        })

log.info(f"  Escalations: {len(escalations)}")
for e in escalations[:5]:
    log.info(f"    {e['item_name']}: warning {e['current_warning']} → {e['next_warning']}")
if len(escalations) > 5:
    log.info(f"    ... and {len(escalations) - 5} more")

log.info(f"  Deletion candidates: {len(deletion_candidates)}")
for d in deletion_candidates[:5]:
    log.info(f"    {d['item_name']} ({d['item_type']})")


## 6. Apply actions to tracker

In [ ]:
# ── Start with current tracker as base ──────────────────
if df_tracker.empty:
    tracker_rows = []
else:
    tracker_rows = df_tracker.to_dict("records")

# Build lookup by item_id for updates
tracker_by_id = {}
for i, row in enumerate(tracker_rows):
    tracker_by_id[str(row["item_id"])] = i

audit_entries = []

# ── 6a. Mark resolved items ───────────────────────────
for r in resolved_items:
    idx = tracker_by_id.get(r["item_id"])
    if idx is not None:
        tracker_rows[idx]["status"] = "resolved"
        tracker_rows[idx]["resolved_date"] = run_timestamp
        tracker_rows[idx]["last_updated"] = run_timestamp
        tracker_rows[idx]["pipeline_run_id"] = pipeline_run_id

        audit_entries.append({
            "audit_id": str(uuid.uuid4()),
            "timestamp": run_timestamp,
            "pipeline_run_id": pipeline_run_id,
            "item_id": r["item_id"],
            "item_name": r["item_name"],
            "item_type": r["item_type"],
            "owner_email": r["owner_email"],
            "action": "owner_resolved",
            "detail": r["reason"],
            "workspace_id": tracker_rows[idx].get("workspace_id", ""),
        })

log.info(f"Applied {len(resolved_items)} resolutions")

# ── 6b. Add new candidates ────────────────────────────
for nc in new_candidates:
    new_row = {
        "item_id": nc["item_id"],
        "item_name": nc["item_name"],
        "item_type": nc["item_type"],
        "owner_email": nc["owner_email"],
        "cleanup_score": nc["cleanup_score"],
        "first_flagged_date": run_timestamp,
        "warning_count": 0,
        "warning_1_date": None,
        "warning_2_date": None,
        "warning_3_date": None,
        "status": "new",
        "deleted_date": None,
        "resolved_date": None,
        "exemption_reason": None,
        "last_updated": run_timestamp,
        "pipeline_run_id": pipeline_run_id,
        "workspace_id": nc["workspace_id"],
    }
    tracker_rows.append(new_row)

    audit_entries.append({
        "audit_id": str(uuid.uuid4()),
        "timestamp": run_timestamp,
        "pipeline_run_id": pipeline_run_id,
        "item_id": nc["item_id"],
        "item_name": nc["item_name"],
        "item_type": nc["item_type"],
        "owner_email": nc["owner_email"],
        "action": "candidate_flagged",
        "detail": f"Score: {nc['cleanup_score']}",
        "workspace_id": nc["workspace_id"],
    })

log.info(f"Added {len(new_candidates)} new candidates")

# ── 6c. Apply escalations ─────────────────────────────
for esc in escalations:
    idx = tracker_by_id.get(esc["item_id"])
    if idx is not None:
        nw = esc["next_warning"]
        tracker_rows[idx]["warning_count"] = nw
        tracker_rows[idx][f"warning_{nw}_date"] = run_timestamp
        tracker_rows[idx]["status"] = esc["next_status"]
        tracker_rows[idx]["last_updated"] = run_timestamp
        tracker_rows[idx]["pipeline_run_id"] = pipeline_run_id

        # Update score from latest snapshot
        snap_match = df_snapshot[df_snapshot["id"].astype(str) == esc["item_id"]]
        if not snap_match.empty:
            current_score = snap_match.iloc[0].get("cleanup_candidate_score", 0)
            if pd.notna(current_score):
                tracker_rows[idx]["cleanup_score"] = int(current_score)

        audit_entries.append({
            "audit_id": str(uuid.uuid4()),
            "timestamp": run_timestamp,
            "pipeline_run_id": pipeline_run_id,
            "item_id": esc["item_id"],
            "item_name": esc["item_name"],
            "item_type": esc["item_type"],
            "owner_email": esc["owner_email"],
            "action": f"warning_{nw}_sent",
            "detail": f"Score: {esc['cleanup_score']}, escalated from warning {esc['current_warning']}",
            "workspace_id": esc["workspace_id"],
        })

log.info(f"Applied {len(escalations)} escalations")

# ── 6d. Mark deletion candidates ──────────────────────
for dc in deletion_candidates:
    idx = tracker_by_id.get(dc["item_id"])
    if idx is not None:
        if enable_auto_delete and not dry_run:
            tracker_rows[idx]["status"] = "pending_deletion"
        else:
            tracker_rows[idx]["status"] = "deletion_ready"
        tracker_rows[idx]["last_updated"] = run_timestamp
        tracker_rows[idx]["pipeline_run_id"] = pipeline_run_id

        audit_entries.append({
            "audit_id": str(uuid.uuid4()),
            "timestamp": run_timestamp,
            "pipeline_run_id": pipeline_run_id,
            "item_id": dc["item_id"],
            "item_name": dc["item_name"],
            "item_type": dc["item_type"],
            "owner_email": dc["owner_email"],
            "action": "deletion_ready" if (not enable_auto_delete or dry_run) else "pending_deletion",
            "detail": f"Score: {dc['cleanup_score']}, past {warnings_before_del} warnings",
            "workspace_id": dc["workspace_id"],
        })

log.info(f"Marked {len(deletion_candidates)} for deletion")


## 7. Save updated tracker

In [ ]:
# ── Write updated tracker (full rewrite: TRUNCATE + bulk INSERT) ───────
df_updated_tracker = pd.DataFrame(tracker_rows)

if not df_updated_tracker.empty:
    # Ensure correct types
    for col in ["warning_count", "cleanup_score"]:
        if col in df_updated_tracker.columns:
            df_updated_tracker[col] = pd.to_numeric(df_updated_tracker[col], errors="coerce").fillna(0).astype(int)

    wh_cursor = conn.cursor()
    wh_cursor.execute(f"TRUNCATE TABLE {GOVERNANCE_SCHEMA}.cleanup_tracker")
    wh_cursor.fast_executemany = True
    cols = list(df_updated_tracker.columns)
    insert_sql = (
        f"INSERT INTO {GOVERNANCE_SCHEMA}.cleanup_tracker ({','.join('[' + c + ']' for c in cols)}) "
        f"VALUES ({','.join(['?']*len(cols))})"
    )
    # Force plain VARCHAR(MAX) binding — item_name/exemption_reason can be long enough that
    # pyodbc auto-detects them as a legacy LOB type, which Fabric Warehouse's UTF-8 collation
    # rejects.
    wh_cursor.setinputsizes([(pyodbc.SQL_VARCHAR, 0, 0)] * len(cols))
    wh_cursor.executemany(insert_sql, df_updated_tracker[cols].astype(str).values.tolist())
    conn.commit()
    log.info(f"✔ cleanup_tracker saved: {len(df_updated_tracker)} rows")
else:
    log.info("No tracker data to save.")

# ── Write audit log entries (append) ──────────────────
if audit_entries:
    df_audit = pd.DataFrame(audit_entries)
    wh_cursor = conn.cursor()
    wh_cursor.fast_executemany = True
    cols = list(df_audit.columns)
    insert_sql = (
        f"INSERT INTO {GOVERNANCE_SCHEMA}.cleanup_audit_log ({','.join('[' + c + ']' for c in cols)}) "
        f"VALUES ({','.join(['?']*len(cols))})"
    )
    # Force plain VARCHAR(MAX) binding — item_name/detail can be long enough to trip pyodbc's
    # legacy-LOB auto-detection under Fabric Warehouse's UTF-8 collation.
    wh_cursor.setinputsizes([(pyodbc.SQL_VARCHAR, 0, 0)] * len(cols))
    wh_cursor.executemany(insert_sql, df_audit[cols].astype(str).values.tolist())
    conn.commit()
    log.info(f"✔ cleanup_audit_log: {len(audit_entries)} entries appended")
else:
    log.info("No audit entries to write.")


## 8. Summary

In [ ]:
print("=" * 60)
print("  GOVERNANCE TRACKER UPDATE — SUMMARY")
print("=" * 60)
print(f"  Pipeline run:     {pipeline_run_id}")
print(f"  Timestamp:        {run_timestamp}")
print(f"  Dry run:          {dry_run}")
print(f"  ")
print(f"  SNAPSHOT")
print(f"  ────────")
print(f"  Items in snapshot:     {len(df_snapshot)}")
print(f"  Score >= {score_threshold}:           {(df_snapshot['cleanup_candidate_score'] >= score_threshold).sum()}")
print(f"  ")
print(f"  ACTIONS THIS RUN")
print(f"  ────────────────")
print(f"  New candidates added:  {len(new_candidates)}")
print(f"  Items resolved:        {len(resolved_items)}")
print(f"  Warnings escalated:    {len(escalations)}")
print(f"  Deletion candidates:   {len(deletion_candidates)}")
print(f"  Skipped (protected):   {skipped_protected}")
print(f"  Skipped (active guard):{skipped_active_guard}")
print(f"  ")
print(f"  TRACKER STATE")
print(f"  ─────────────")
if not df_updated_tracker.empty:
    for status, count in df_updated_tracker["status"].value_counts().items():
        print(f"  {status:25s} {count}")
    print(f"  {'─' * 35}")
    print(f"  {'Total':25s} {len(df_updated_tracker)}")
print(f"  ")
print(f"  AUDIT LOG")
print(f"  ─────────")
print(f"  Entries written:       {len(audit_entries)}")
print("=" * 60)


## 9. View current tracker state

In [ ]:
df_view = pd.read_sql(f"""
    SELECT
        status,
        COUNT(*) as item_count,
        AVG(CAST(cleanup_score AS INT)) as avg_score
    FROM {GOVERNANCE_SCHEMA}.cleanup_tracker
    GROUP BY status
    ORDER BY item_count DESC
""", conn)
display(df_view)

In [ ]:
df_view = pd.read_sql(f"""
    -- Items approaching deletion (warning_3 or deletion_ready)
    SELECT item_name, item_type, owner_email, CAST(cleanup_score AS INT) as score,
           CAST(warning_count AS INT) as warnings, status,
           warning_1_date, warning_2_date, warning_3_date
    FROM {GOVERNANCE_SCHEMA}.cleanup_tracker
    WHERE status IN ('warning_3', 'deletion_ready', 'pending_deletion')
    ORDER BY CAST(cleanup_score AS INT) DESC
""", conn)
display(df_view)

In [ ]:
df_view = pd.read_sql(f"""
    -- Latest audit log entries
    SELECT TOP 20 [timestamp], item_name, item_type, action, detail
    FROM {GOVERNANCE_SCHEMA}.cleanup_audit_log
    ORDER BY [timestamp] DESC
""", conn)
display(df_view)